# 

# Generalization Datasets

- Classificaiton
  - tox21
  - toxcast
  - muv
  - pcba
- Regression
  - hopv - homo, lumo
  - zinc15 - logp
  - freesolv - hydration free energy
- Rxn
  - open reaction database
    - presto dataset( not compare with presto, because we assume OOD comparison)
- M2T
  - hanbum's dataset
- T2M
  - hanbum's dataset

# Check dataset availability

In [1]:
from ogb.utils.features import (allowable_features, atom_to_feature_vector,
 bond_to_feature_vector, atom_feature_vector_to_dict, bond_feature_vector_to_dict) 
import numpy as np
from tqdm import tqdm
import instructions_smol
import datasets
from datasets import load_dataset
import pandas as pd
import os
from rdkit import Chem


def mol2graph(mol):
    """
    Converts SMILES string to graph Data object
    :input: SMILES string (str)
    :return: graph object
    """
    # atoms
    atom_features_list = []
    for atom in mol.GetAtoms():
        atom_features_list.append(atom_to_feature_vector(atom))
    x = np.array(atom_features_list, dtype = np.int64)

    # bonds
    num_bond_features = 3  # bond type, bond stereo, is_conjugated
    if len(mol.GetBonds()) > 0: # mol has bonds
        edges_list = []
        edge_features_list = []
        for bond in mol.GetBonds():
            i = bond.GetBeginAtomIdx()
            j = bond.GetEndAtomIdx()

            edge_feature = bond_to_feature_vector(bond)

            # add edges in both directions
            edges_list.append((i, j))
            edge_features_list.append(edge_feature)
            edges_list.append((j, i))
            edge_features_list.append(edge_feature)

        # data.edge_index: Graph connectivity in COO format with shape [2, num_edges]
        edge_index = np.array(edges_list, dtype = np.int64).T

        # data.edge_attr: Edge feature matrix with shape [num_edges, num_edge_features]
        edge_attr = np.array(edge_features_list, dtype = np.int64)

    else:   # mol has no bonds
        edge_index = np.empty((2, 0), dtype = np.int64)
        edge_attr = np.empty((0, num_bond_features), dtype = np.int64)

    graph = dict()
    graph['edge_index'] = edge_index
    graph['edge_feat'] = edge_attr
    graph['node_feat'] = x
    graph['num_nodes'] = len(x)

    return graph 

from rdkit import Chem
import selfies as sf
from download_dataset import wrap_label

system_prompt = "You are a helpful assistant for molecular chemistry, to address tasks including molecular property classification, molecular property regression, chemical reaction prediction, molecule captioning, molecule generation."

def prepare_data_instance(
        mol,
        label,
        task,
        instruction_templates,
        system_prompt,
        mol_token="<mol>",
        num_query_tokens=32,
):

    label = wrap_label(label, task=task)
    input_prompt = np.random.choice(instruction_templates).item()
    assert "<INPUT>" in input_prompt, f"llm_prompt should contain <INPUT>"
    graph_sequence = "<GRAPH>" + mol_token * num_query_tokens + "</GRAPH>"

    if "reagent_prediction" in task:
        smiles = Chem.MolToSmiles(mol[0])
        selfies = sf.encoder(smiles)
        input_mol_string = "<SELFIES> " + selfies + " </SELFIES>"
        input_mol_string_graph = input_mol_string + graph_sequence

        additional_smiles = Chem.MolToSmiles(mol[1])
        additional_selfies = sf.encoder(additional_smiles)
        additional_input_mol_string = "<SELFIES> " + additional_selfies + " </SELFIES>"
        additional_input_mol_string_graph = additional_input_mol_string + graph_sequence

        input_mol_string = input_mol_string + "|>>|" + additional_input_mol_string
        input_mol_string_graph = input_mol_string_graph + "|>>|" + additional_input_mol_string_graph
        
        graph = mol2graph(mol[0])
        additional_graph = mol2graph(mol[1])
    else:
        assert isinstance(mol, Chem.Mol), f"mol should be a RDKit Mol object, but got {type(mol)}"
        mol = mol
        smiles = Chem.MolToSmiles(mol)
        selfies = sf.encoder(smiles)
        input_mol_string = "<SELFIES> " + selfies + " </SELFIES>"
        input_mol_string_graph = input_mol_string + graph_sequence

        graph = mol2graph(mol)
        additional_graph = graph
    
    input_prompt = input_prompt.replace("<INPUT>", input_mol_string_graph)

    formatted_prompt_text = "<s>[INST] " + system_prompt + " \n\n" + input_prompt + " [INST]"
    formatted_target_text = label + " </s>"


    data = {
        "task": task,
        "x": graph['node_feat'],
        "edge_index": graph['edge_index'],
        "edge_attr": graph['edge_feat'],
        "additional_x": additional_graph['node_feat'],
        "additional_edge_index": additional_graph['edge_index'],
        "additional_edge_attr": additional_graph['edge_feat'],
        "input_mol_string": input_mol_string,
        "prompt_text": formatted_prompt_text,
        "target_text": formatted_target_text,
    }
    return data


def get_data_list(
        list_mol, list_label, task, instruction_templates, system_prompt
):
    list_data = []
    iter_bar = tqdm(range(len(list_mol)))

    for i in iter_bar:
        data = prepare_data_instance(
        mol=list_mol[i],
        label=list_label[i], 
        task=task,
        instruction_templates=instruction_templates,
        system_prompt=system_prompt
    )  
        list_data.append(data)
    return list_data


/miniconda/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
No normalization for SPS. Feature removed!
No normalization for AvgIpc. Feature removed!
Skipped loading some Tensorflow models, missing a dependency. No module named 'tensorflow'
Skipped loading modules with pytorch-geometric dependency, missing a dependency. No module named 'dgl'
Skipped loading modules with pytorch-lightning dependency, missing a dependency. No module named 'lightning'
Skipped loading some Jax models, missing a dependency. No module named 'jax'
Skipped loading some PyTorch models, missing a dependency. No module named 'tensorflow'


In [2]:

rp_data = load_dataset("OpenMol/RCR_RP_57K_SMILES-MMChat")
task_name = "presto-reagent_prediction"
instruction_templates = instructions_smol.reagent_prediction

list_mol = []
list_label = []
omitted_idx = []    
from tqdm import tqdm
iter_bar = tqdm(range(len(rp_data['test'])))
for i in iter_bar:
    smiles = rp_data['test'][i]['molecules']['smiles']
    label = rp_data['test'][i]['ground_truth']
    if isinstance(smiles, list) and len(smiles) > 1:
        mol = [Chem.MolFromSmiles(s) for s in smiles]
    else:
        mol = Chem.MolFromSmiles(smiles[0])
    try:
        label = sf.encoder(label)
        list_label.append(label)
        list_mol.append(mol)
    except:
        omitted_idx.append(i)
        continue
    
list_rp_data = get_data_list(
    list_mol=list_mol,
    list_label=list_label,
    task=task_name,
    instruction_templates=instruction_templates,
    system_prompt=system_prompt
)
print(len(list_rp_data), len(omitted_idx))
rp_dataset = datasets.Dataset.from_list(list_rp_data)

llm_model = "mistralai/Mistral-7B-Instruct-v0.3"
mol_representation = "string+graph"
num_query_token = 32
base_model = llm_model.replace("/", "-")
tags = [base_model, mol_representation]
if "graph" in mol_representation:
    tags += [f"q{num_query_token}"]

processed_file_name = "_".join(tags)
raw_data_root = "/data/data/Mol-LLM-v7.1"

rp_dataset.save_to_disk(f"{raw_data_root}/{processed_file_name}_train_{task_name}")
rp_dataset.save_to_disk(f"{raw_data_root}/{processed_file_name}_test_{task_name}")
rp_dataset.save_to_disk(f"{raw_data_root}/{processed_file_name}_validation_{task_name}")

print(rp_dataset[0])

100%|██████████| 6377/6377 [00:12<00:00, 530.62it/s]


6377 1


Saving the dataset (1/1 shards): 100%|██████████| 6377/6377 [00:00<00:00, 109148.58 examples/s]

{'task': 'presto-reagent_prediction', 'x': [[5, 0, 4, 5, 3, 0, 2, 0, 0], [5, 0, 4, 5, 2, 0, 2, 0, 0], [7, 0, 2, 5, 0, 0, 1, 0, 0], [5, 0, 3, 5, 0, 0, 1, 0, 0], [7, 0, 1, 5, 0, 0, 1, 0, 0], [5, 0, 3, 5, 0, 0, 1, 1, 1], [5, 0, 3, 5, 1, 0, 1, 1, 1], [5, 0, 3, 5, 1, 0, 1, 1, 1], [5, 0, 3, 5, 0, 0, 1, 1, 1], [5, 0, 3, 5, 0, 0, 1, 1, 1], [5, 0, 3, 5, 0, 0, 1, 1, 1], [7, 0, 2, 5, 0, 0, 1, 0, 0], [5, 0, 4, 5, 3, 0, 2, 0, 0], [5, 0, 3, 5, 1, 0, 1, 1, 1], [5, 0, 3, 5, 1, 0, 1, 1, 1], [5, 0, 3, 5, 1, 0, 1, 1, 1], [5, 0, 3, 5, 0, 0, 1, 1, 1], [7, 0, 2, 5, 0, 0, 1, 0, 0], [5, 0, 4, 5, 3, 0, 2, 0, 0], [5, 0, 3, 5, 0, 0, 1, 1, 1], [5, 0, 3, 5, 0, 0, 1, 1, 1], [5, 0, 3, 5, 1, 0, 1, 1, 1], [5, 0, 3, 5, 1, 0, 1, 1, 1], [5, 0, 3, 5, 0, 0, 1, 1, 1], [16, 0, 1, 5, 0, 0, 2, 0, 0], [5, 0, 3, 5, 0, 0, 1, 1, 1], [7, 0, 2, 5, 0, 0, 1, 0, 0], [5, 0, 4, 5, 2, 0, 2, 0, 0], [5, 0, 4, 5, 2, 0, 2, 0, 0], [5, 0, 4, 5, 2, 0, 2, 0, 0], [6, 0, 3, 5, 0, 0, 2, 0, 0], [5, 0, 4, 5, 3, 0, 2, 0, 0], [5, 0, 4, 5, 3, 0, 2, 0, 0]

In [3]:
fs_data = load_dataset("OpenMol/MolInst_FS_125K_Scaffold_SMILES-MMChat")
task_name = "presto-forward_reaction_prediction"
instruction_templates = instructions_smol.forward_reaction_prediction

list_mol = []
list_label = []
omitted_idx = []    
from tqdm import tqdm
iter_bar = tqdm(range(len(fs_data['test'])))
for i in iter_bar:
    smiles = fs_data['test'][i]['molecules']['smiles']
    label = fs_data['test'][i]['ground_truth']
    if isinstance(smiles, list):
        combined_smiles = '.'.join(smiles)
        mol = Chem.MolFromSmiles(combined_smiles)
    else:
        raise ValueError("smiles should be a list")
    try:
        label = sf.encoder(label)
        list_label.append(label)
        list_mol.append(mol)
    except:
        omitted_idx.append(i)
        continue

list_fs_data = get_data_list(
    list_mol=list_mol,
    list_label=list_label,
    task=task_name,
    instruction_templates=instruction_templates,
    system_prompt=system_prompt
)
print(len(list_fs_data), len(omitted_idx))
fs_dataset = datasets.Dataset.from_list(list_fs_data)

llm_model = "mistralai/Mistral-7B-Instruct-v0.3"
mol_representation = "string+graph"
num_query_token = 32
base_model = llm_model.replace("/", "-")
tags = [base_model, mol_representation]
if "graph" in mol_representation:
    tags += [f"q{num_query_token}"]

processed_file_name = "_".join(tags)
raw_data_root = "/data/data/Mol-LLM-v7.1"

fs_dataset.save_to_disk(f"{raw_data_root}/{processed_file_name}_train_{task_name}")
fs_dataset.save_to_disk(f"{raw_data_root}/{processed_file_name}_test_{task_name}")
fs_dataset.save_to_disk(f"{raw_data_root}/{processed_file_name}_validation_{task_name}")

print(fs_dataset[0])

  0%|          | 0/1004 [00:00<?, ?it/s][05:48:35] WARNING: not removing hydrogen atom without neighbors
[05:48:35] WARNING: not removing hydrogen atom without neighbors
[05:48:35] WARNING: not removing hydrogen atom without neighbors
[05:48:35] WARNING: not removing hydrogen atom without neighbors
[05:48:35] WARNING: not removing hydrogen atom without neighbors
[05:48:35] WARNING: not removing hydrogen atom without neighbors
[05:48:35] WARNING: not removing hydrogen atom without neighbors
[05:48:35] WARNING: not removing hydrogen atom without neighbors
 13%|█▎        | 130/1004 [00:00<00:00, 1288.57it/s][05:48:35] WARNING: not removing hydrogen atom without neighbors
[05:48:35] WARNING: not removing hydrogen atom without neighbors
[05:48:35] WARNING: not removing hydrogen atom without neighbors
[05:48:35] WARNING: not removing hydrogen atom without neighbors
[05:48:35] WARNING: not removing hydrogen atom without neighbors
[05:48:35] WARNING: not removing hydrogen atom without neighbor

1004 0


Saving the dataset (1/1 shards): 100%|██████████| 1004/1004 [00:00<00:00, 51850.39 examples/s]

{'task': 'presto-forward_reaction_prediction', 'x': [[7, 0, 1, 5, 0, 0, 1, 0, 0], [5, 0, 3, 5, 0, 0, 1, 0, 0], [16, 0, 1, 5, 0, 0, 2, 0, 0], [5, 0, 3, 5, 0, 0, 1, 0, 0], [7, 0, 1, 5, 0, 0, 1, 0, 0], [16, 0, 1, 5, 0, 0, 2, 0, 0], [7, 0, 1, 5, 0, 0, 1, 0, 0], [5, 0, 3, 5, 0, 0, 1, 0, 0], [7, 0, 2, 5, 1, 0, 1, 0, 0], [5, 0, 4, 5, 2, 0, 2, 0, 0], [5, 0, 4, 5, 2, 0, 2, 0, 0], [5, 0, 4, 5, 0, 0, 2, 0, 0], [8, 0, 1, 5, 0, 0, 2, 0, 0], [8, 0, 1, 5, 0, 0, 2, 0, 0], [8, 0, 1, 5, 0, 0, 2, 0, 0], [5, 0, 4, 5, 3, 0, 2, 0, 0], [6, 0, 3, 5, 0, 0, 1, 0, 0], [5, 0, 4, 5, 3, 0, 2, 0, 0], [5, 0, 3, 5, 1, 0, 1, 0, 0], [7, 0, 1, 5, 0, 0, 1, 0, 0], [16, 0, 1, 5, 0, 0, 2, 0, 0], [5, 0, 4, 5, 2, 0, 2, 0, 0], [16, 0, 1, 5, 0, 0, 2, 0, 0]], 'edge_index': [[0, 1, 1, 2, 1, 3, 3, 4, 3, 5, 6, 7, 7, 8, 7, 9, 9, 10, 10, 11, 11, 12, 11, 13, 11, 14, 15, 16, 16, 17, 16, 18, 18, 19, 20, 21, 21, 22], [1, 0, 2, 1, 3, 1, 4, 3, 5, 3, 7, 6, 8, 7, 9, 7, 10, 9, 11, 10, 12, 11, 13, 11, 14, 11, 16, 15, 17, 16, 18, 16, 19, 18, 21,

In [4]:
rs_data = load_dataset("OpenMol/MolInst_RS_125K_Scaffold_SMILES-MMChat")
task_name = "presto-retrosynthesis"
instruction_templates = instructions_smol.retrosynthesis

list_mol = []
list_label = []
omitted_idx = []    
from tqdm import tqdm
iter_bar = tqdm(range(len(fs_data['test'])))
for i in iter_bar:
    smiles = fs_data['test'][i]['molecules']['smiles']
    label = fs_data['test'][i]['ground_truth']
    if isinstance(smiles, list):
        combined_smiles = '.'.join(smiles)
        mol = Chem.MolFromSmiles(combined_smiles)
    else:
        raise ValueError("smiles should be a list")
    try:
        label = sf.encoder(label)
        list_label.append(label)
        list_mol.append(mol)
    except:
        omitted_idx.append(i)
        continue

list_rs_data = get_data_list(
    list_mol=list_mol,
    list_label=list_label,
    task=task_name,
    instruction_templates=instruction_templates,
    system_prompt=system_prompt
)
print(len(list_rs_data), len(omitted_idx))
rs_dataset = datasets.Dataset.from_list(list_rs_data)

llm_model = "mistralai/Mistral-7B-Instruct-v0.3"
mol_representation = "string+graph"
num_query_token = 32
base_model = llm_model.replace("/", "-")
tags = [base_model, mol_representation]
if "graph" in mol_representation:
    tags += [f"q{num_query_token}"]

processed_file_name = "_".join(tags)
raw_data_root = "/data/data/Mol-LLM-v7.1"

rs_dataset.save_to_disk(f"{raw_data_root}/{processed_file_name}_train_{task_name}")
rs_dataset.save_to_disk(f"{raw_data_root}/{processed_file_name}_test_{task_name}")
rs_dataset.save_to_disk(f"{raw_data_root}/{processed_file_name}_validation_{task_name}")

print(rs_dataset[0])

  0%|          | 0/1004 [00:00<?, ?it/s][05:48:40] WARNING: not removing hydrogen atom without neighbors
[05:48:40] WARNING: not removing hydrogen atom without neighbors
[05:48:40] WARNING: not removing hydrogen atom without neighbors
[05:48:40] WARNING: not removing hydrogen atom without neighbors
[05:48:40] WARNING: not removing hydrogen atom without neighbors
[05:48:40] WARNING: not removing hydrogen atom without neighbors
[05:48:40] WARNING: not removing hydrogen atom without neighbors
[05:48:40] WARNING: not removing hydrogen atom without neighbors
 13%|█▎        | 128/1004 [00:00<00:00, 1268.60it/s][05:48:40] WARNING: not removing hydrogen atom without neighbors
[05:48:40] WARNING: not removing hydrogen atom without neighbors
[05:48:40] WARNING: not removing hydrogen atom without neighbors
[05:48:40] WARNING: not removing hydrogen atom without neighbors
[05:48:40] WARNING: not removing hydrogen atom without neighbors
[05:48:40] WARNING: not removing hydrogen atom without neighbor

1004 0


Saving the dataset (1/1 shards): 100%|██████████| 1004/1004 [00:00<00:00, 50648.05 examples/s]

{'task': 'presto-retrosynthesis', 'x': [[7, 0, 1, 5, 0, 0, 1, 0, 0], [5, 0, 3, 5, 0, 0, 1, 0, 0], [16, 0, 1, 5, 0, 0, 2, 0, 0], [5, 0, 3, 5, 0, 0, 1, 0, 0], [7, 0, 1, 5, 0, 0, 1, 0, 0], [16, 0, 1, 5, 0, 0, 2, 0, 0], [7, 0, 1, 5, 0, 0, 1, 0, 0], [5, 0, 3, 5, 0, 0, 1, 0, 0], [7, 0, 2, 5, 1, 0, 1, 0, 0], [5, 0, 4, 5, 2, 0, 2, 0, 0], [5, 0, 4, 5, 2, 0, 2, 0, 0], [5, 0, 4, 5, 0, 0, 2, 0, 0], [8, 0, 1, 5, 0, 0, 2, 0, 0], [8, 0, 1, 5, 0, 0, 2, 0, 0], [8, 0, 1, 5, 0, 0, 2, 0, 0], [5, 0, 4, 5, 3, 0, 2, 0, 0], [6, 0, 3, 5, 0, 0, 1, 0, 0], [5, 0, 4, 5, 3, 0, 2, 0, 0], [5, 0, 3, 5, 1, 0, 1, 0, 0], [7, 0, 1, 5, 0, 0, 1, 0, 0], [16, 0, 1, 5, 0, 0, 2, 0, 0], [5, 0, 4, 5, 2, 0, 2, 0, 0], [16, 0, 1, 5, 0, 0, 2, 0, 0]], 'edge_index': [[0, 1, 1, 2, 1, 3, 3, 4, 3, 5, 6, 7, 7, 8, 7, 9, 9, 10, 10, 11, 11, 12, 11, 13, 11, 14, 15, 16, 16, 17, 16, 18, 18, 19, 20, 21, 21, 22], [1, 0, 2, 1, 3, 1, 4, 3, 5, 3, 7, 6, 8, 7, 9, 7, 10, 9, 11, 10, 12, 11, 13, 11, 14, 11, 16, 15, 17, 16, 18, 16, 19, 18, 21, 20, 22, 21]]